In [14]:
from pathlib import Path
from decouple import AutoConfig

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_classic.chains import RetrievalQA
from pinecone import Pinecone, ServerlessSpec
import zipfile
import os

In [19]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
pinecone_api_key = config("PINECONE_API_KEY")
openai_api_key = config("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = pinecone_api_key

In [ ]:
pinecone_client = Pinecone(api_key=pinecone_api_key)
pinecone_client.create_index(
    name="llm",
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

In [8]:
zip_file_path = 'documentos.zip'
extracted_folder_path = 'docs'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_folder_path)

documents = []
for filename in os.listdir(extracted_folder_path):
    if filename.endswith(".pdf"):
        file_path = os.path.join(extracted_folder_path, filename)
        loader = PyMuPDFLoader(file_path)
        documents.extend(loader.load())


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)
chunks = text_splitter.create_documents([doc.page_content for doc in documents])

In [12]:
embeddings = OpenAIEmbeddings(api_key=openai_api_key,model='text-embedding-ada-002')

In [20]:
index_name = 'llm'
vector_store = PineconeVectorStore.from_documents(
    chunks,
    embeddings,
    index_name=index_name
)

In [21]:
query1 = '''
Responda apenas com base no input fornecido. Qual o número do processo que trata de Violaçāo de normas ambientais pela empresa de Construçāo?
'''
query2 = 'Responda apenas com base no input fornecido. Qual foi a decisāo no caso de faude financeira?'
query3 = 'Responda apenas com base no input fornecido.Quais foram as alegaçōes no caso de negligência médica?'
query4 = 'Responda apenas com base no input fornecido.Quais foram as alegaçōes no caso de Número do Precsso: 822161'

In [22]:
llm = ChatOpenAI(
    api_key=openai_api_key,
    model='gpt-4o-mini',
    store=False,
    temperature=0.2
)

In [24]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k': 3
    }
)

In [25]:
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever
)

In [26]:
answer_1 = chain.invoke(query1)
answer_2 = chain.invoke(query2)
answer_3 = chain.invoke(query3)
answer_4 = chain.invoke(query4)

In [27]:
print(f'Pergunta: {answer_1['query']}')
print(f'Resultado: {answer_1['result']} \n')

print(f'Pergunta: {answer_2['query']}')
print(f'Resultado: {answer_2['result']} \n')


print(f'Pergunta: {answer_3['query']}')
print(f'Resultado: {answer_3['result']} \n')

print(f'Pergunta: {answer_4['query']}')
print(f'Resultado: {answer_4['result']} \n')

Pergunta: 
Responda apenas com base no input fornecido. Qual o número do processo que trata de Violaçāo de normas ambientais pela empresa de Construçāo?

Resultado: O número do processo que trata de violação de normas ambientais pela empresa de construção é 175543. 

Pergunta: Responda apenas com base no input fornecido. Qual foi a decisāo no caso de faude financeira?
Resultado: A decisão no caso de fraude financeira envolvendo João Almeida foi de dez anos de prisão e a restituição de $1,000,000 ao Banco Nacional. 

Pergunta: Responda apenas com base no input fornecido.Quais foram as alegaçōes no caso de negligência médica?
Resultado: As alegações no caso de negligência médica foram que Dr. Pedro Sousa não realizou os procedimentos médicos com o cuidado necessário, resultando em complicações graves, incluindo uma cirurgia mal-sucedida que levou a infecções adicionais. Depoimentos de especialistas médicos apoiaram essas alegações de negligência. 

Pergunta: Responda apenas com base no i